In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import *
from pyspark.sql.functions import *

In [0]:
rows = [
    Row(
        id=1,
        names=["Alice", "Bob"],
        scores=[85, 90],
        info=Row(age=25, city="New York"),
        json_str='{"dept": "Engineering", "level": 2}'
    ),
    Row(
        id=2,
        names=["Carol"],
        scores=[78, 88, 92],
        info=Row(age=30, city="San Francisco"),
        json_str='{"dept": "HR", "level": 1}'
    )
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("names", ArrayType(StringType()), True),
    StructField("scores", ArrayType(IntegerType()), True),
    StructField("info", StructType([
        StructField("age", IntegerType(), True),
        StructField("city", StringType(), True)
    ]), True),
    StructField("json_str", StringType(), True)
])

df = spark.createDataFrame(rows, schema)
df.createOrReplaceTempView("df")
display(df)

In [0]:
from pyspark.sql import functions as F
array_length_df = df.withColumn("namelenght",F.size(F.col("names")))
display(array_length_df)

In [0]:
array_contains_df = array_length_df.withColumn("contains_alice",F.array_contains(F.col("names"),"Alice"))
display(array_contains_df)

In [0]:
df_array_split = array_contains_df.withColumn("split_names",F.split(F.col("info.city")," "))
display(df_array_split)

In [0]:
display(df_array_split.select("id","names","scores","info","json_str","namelenght","contains_alice","split_names"))

In [0]:
%sql
select id,
names,
scores,
size(names) as length,
array_contains(names,"Alice") as has_alice,
split(info.city,' ') as city_split,
get_json_object(json_str, '$.dept') AS dept
from df

In [0]:
# MAGIC %md
# MAGIC ### Exploding Arrays
# MAGIC
# MAGIC The `explode` function creates a new row for each element in an array column. This is useful for flattening nested data.
explode_df = df.withColumn("exploded_names",F.explode(F.col("names").alias("names")))
display(explode_df)

In [0]:
# Accessing fields in the 'info' struct
info_df = df.withColumn("cityinfo" , F.col("info.city")).withColumn("ageinfo",F.col("info.age"))
display(info_df)

In [0]:
json_df = df.withColumn("dept",F.get_json_object(F.col("json_str"),"$.dept")) \
    .withColumn("level",F.get_json_object(F.col("json_str"),"$.level"))
display(json_df)    

In [0]:
names_df =df.withColumn("explode_name",F.explode(F.col("names")))
display(names_df)

In [0]:
# Parse JSON string into struct
json_schema = StructType([
    StructField("dept", StringType(), True),
    StructField("level", IntegerType(), True)
])
from_json_df = df.withColumn("json_struct", F.from_json(F.col("json_str"), json_schema))
# Accessing fields in the 'info' struct
struct_df = from_json_df.withColumn("dept", F.col("json_struct.dept"))

display(struct_df)